# 03 — Customer segmentation, churn and product matching

This notebook inspects the RFM segmentation and supervised churn workflow used by AutoDS. Churn labels are behavioural inactivity proxies, not confirmed cancellations, and the churn feature set excludes recency to avoid direct target leakage.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "backend").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import pandas as pd

from app.database.base import SessionLocal, init_db
from app.services.data_service import ensure_demo_dataset, get_context

init_db()
db = SessionLocal()
dataset = ensure_demo_dataset(db)
ctx = get_context(dataset.id, db)
print(f"Customer analysis through {ctx.date_max}")

## RFM segmentation

K-Means uses standardized RFM, average order value, purchase frequency and tenure. The silhouette score and Davies–Bouldin index expose cluster quality; segment labels are assigned from cluster profiles.

In [ ]:
from app.services.customer_service import segment_customers

segmentation = segment_customers(ctx, limit=20)
pd.Series(segmentation["metrics"], name="value")

In [ ]:
clusters = pd.DataFrame(segmentation["clusters"])
clusters

In [ ]:
top_customers = pd.DataFrame(segmentation["customers"])
top_customers.head(10)

## Churn model

Logistic Regression, Random Forest and XGBoost are compared with cross-validated ROC-AUC on the training split. The selected model is evaluated once on a stratified holdout. Probabilities are decision-support scores, not causal effects.

In [ ]:
from app.services.customer_service import churn_analysis

churn = churn_analysis(ctx, inactive_days=90)
pd.Series(churn["metrics"], name="value")

In [ ]:
risk = pd.DataFrame(churn["risk_distribution"])
importance = pd.DataFrame(churn["feature_importance"])
display(risk)
importance.sort_values("importance", ascending=False).head(10)

In [ ]:
at_risk = pd.DataFrame(churn["customers"])
at_risk[["customer_id", "segment", "recency_days", "frequency", "monetary", "churn_probability", "churn_risk"]].head(12)

## Product matching

Recommendations use the customer's observed category and product history. They identify cross-sell candidates; they are not a guarantee of incremental revenue.

In [ ]:
from app.services.customer_service import recommend_products

customer_id = top_customers.iloc[0]["customer_id"]
recommendations = recommend_products(ctx, customer_id, top_n=5)
print(f"Customer: {customer_id}")
pd.DataFrame(recommendations["recommendations"])

In [ ]:
db.close()